# Experiment One:
This experiment is intended to train networks using various $\psi$ and intrinsic length settings, measure classifiation accuracy on CIFAR10, then perform various transforms, neurogenesis and neurodegeneration, and observe the effect upon the accuracy and average difference magnitude before and after transforms. These will be undertaken with multiple repeats to obtain standard error.

In [ ]:
from Dependencies import *
import torch.nn as nn
import torch
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

In [ ]:
# Hyperparameters
LEARNING_RATE = 1e-3
BATCH_SIZE = 48
DEVICE = try_gpu(output=True, i=0)
ARCHITECTURE = [3072, 100, 100, 10]
REPEATS = 20
PSI_DECAY = 1e-1
ADAMW_WEIGHT_DECAY = 1e-3
TOTAL_EPOCHS = 50
save_dir = "./Saved_Models/Experiment 1/Baseline/"
NORMALISATION = True
os.makedirs(save_dir, exist_ok=True)

In [ ]:
# Found that elementwise normalisation of the dataset was generally beneficial as a preprocessing step
class PerPixelNormalize:
    def __init__(self):
        normaliser_dictionary = pkl.load(open("./CIFAR_normalisations.pkl", "rb"))
        self.mean = normaliser_dictionary["mean"].to(torch.float32)
        self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

    def __call__(self, tensor):
        return (tensor - self.mean) * self.inv_std

In [ ]:
if NORMALISATION:
    print("Using Normalisation")
    transform = transforms.Compose([transforms.ToTensor(), PerPixelNormalize()])
else:
    print("Not using Normalisation")
    transform = transforms.Compose([transforms.ToTensor()])

# Get dataset
cifar_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
# DataLoaders
train_loader = DataLoader(cifar_train, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(cifar_test, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
combinations = [["NONE", "TRAINABLE+DECAY",], ["CONSTANT", "TRAINABLE+DECAY",], ["TRAINABLE", "TRAINABLE+DECAY",],
                ["TRAINABLE", "NONE"], ["TRAINABLE", "CONSTANT"], ["TRAINABLE", "TRAINABLE",], ["TRAINABLE", "TRAINABLE+DECAY",], ["TRAINABLE", "EXPECTATION"]]

for REPEAT in range(REPEATS):
    for intrinsic_length_approach, linear_correction_approach in combinations:
        # Build filename
        save_name = f"repeat_{REPEAT:02d}_{intrinsic_length_approach}_{linear_correction_approach}_{TOTAL_EPOCHS}.pkl"
        save_path = os.path.join(save_dir, save_name)
        if save_name in os.listdir(save_dir): print(f"Skipping {save_name}"); continue
        else: print(f"Training {save_name}")


        network = IsotropicTanhMLP(
            layers = ARCHITECTURE,
            flatten = True, unflatten_shape = None,
            intrinsic_length_approach = intrinsic_length_approach,
            linear_correction_approach = linear_correction_approach,
            positive_intrinsic_length = True,
            init_intrinsic_length = 1e-6,
            tanh_epsilon = 1e-3,
            device = DEVICE, dtype = torch.get_default_dtype(),
            )

        # Initialise network parameters
        network.simple_initialiser(weight_init="orthogonal")

        # Train baseline isotropic network for classification on CIFAR10
        optimiser = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE, weight_decay=ADAMW_WEIGHT_DECAY)
        trained_network, stats = training_loop(
            network=network,
            training_set=train_loader,
            testing_set=test_loader,
            epochs=TOTAL_EPOCHS,
            learning_rate=LEARNING_RATE,
            device=DEVICE,
            quiet=True,
            optimiser = optimiser,
            classification_or_reconstruction="classification",
            lambda_psi=PSI_DECAY if linear_correction_approach == "TRAINABLE+DECAY" else 0.0,
        )

        # Build filename
        save_name_pkl = f"repeat_{REPEAT:02d}_{intrinsic_length_approach}_{linear_correction_approach}_{TOTAL_EPOCHS}.pkl"
        save_name_svg = f"repeat_{REPEAT:02d}_{intrinsic_length_approach}_{linear_correction_approach}_{TOTAL_EPOCHS}.svg"
        save_path_pkl = os.path.join(save_dir, save_name_pkl)
        save_path_svg = os.path.join(save_dir, save_name_svg)

        train_x, test_x, train_cost, test_cost, train_acc, test_acc = stats



        train_x = np.asarray(train_x)
        train_acc = np.asarray(train_acc)
        test_x = np.asarray(test_x)
        test_acc = np.asarray(test_acc)

        # Average training accuracy within each epoch
        epoch_train_x = np.arange(1, TOTAL_EPOCHS + 1)
        epoch_train_acc = np.array([
            train_acc[(train_x >= epoch - 1) & (train_x < epoch)].mean()
            for epoch in epoch_train_x
        ])

        plt.figure(figsize=(8, 5))
        plt.plot(epoch_train_x, epoch_train_acc, label="Train Accuracy")
        plt.plot(test_x, test_acc, label="Test Accuracy")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy (%)")
        plt.ylim(0, 100)
        plt.xlim(0, TOTAL_EPOCHS)
        plt.title("CIFAR-10 Accuracy vs Epoch")
        plt.legend()
        plt.tight_layout()
        plt.savefig(save_path_svg)
        plt.show()



        # Save trained model and statistics

        with open(save_path_pkl, "wb") as f:
            pkl.dump({"stats": stats, "model": trained_network, }, f)

        print(f"Saved to {save_path}")


Then undergo network resizing. Original neuroadaption aspect written by authors, with ChatGPT to format into a table on the go.

In [ ]:
import copy
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from Dependencies import *



LAYER = 1   # For ARCHITECTURE=[3072, 100, 100, 10], full_diagonalise is only valid at layer=1.

# Prefer notebook globals if they already exist
if "REPEATS" not in globals():
    REPEATS = 10
else: print(f"{REPEATS=}")

ROWS = [+75, +50, +25, +5, +1, 0, -1, -5, -25, -50, -75, "baseline"]

COMBINATIONS = [
    ["NONE", "TRAINABLE+DECAY"],
    ["CONSTANT", "TRAINABLE+DECAY"],
    ["TRAINABLE", "TRAINABLE+DECAY"],
    ["TRAINABLE", "NONE"],
    ["TRAINABLE", "CONSTANT"],
    ["TRAINABLE", "TRAINABLE"],
    ["TRAINABLE", "TRAINABLE+DECAY"],
    ["TRAINABLE", "EXPECTATION"],
]

# Match the notebook save directory exactly
SAVE_DIR = Path("./Saved_Models/Experiment 1/Baseline/")

# Prefer notebook global if already defined there
if "TOTAL_EPOCHS" not in globals():
    TOTAL_EPOCHS = 50

# Set to True if you want progress prints
VERBOSE = True

# Output tex path
OUTPUT_TEX = Path(f"resize_eval_layer_{LAYER}.tex")

# -------------------------
# Helpers
# -------------------------
def combo_to_key(combo):
    return f"{combo[0]}__{combo[1]}"

def combo_to_header(combo):
    return f"{combo[0]} + {combo[1]}"

def safe_std(x):
    x = np.asarray(x, dtype=float)
    if x.size <= 1:
        return 0.0
    return np.std(x, ddof=1)

def format_cell(acc_mean, acc_std, eps_mean, eps_std, sv_mean, sv_std,
                row_value, digits_acc=3, digits_eps=4, digits_sv=4):
    if row_value == "baseline":
        return (
            f"\\begin{{tabular}}[c]{{@{{}}c@{{}}}}"
            f"{acc_mean:.{digits_acc}f} $\\pm$ {acc_std:.{digits_acc}f} \\\\ "
            f"$\\sum \\sigma_0$ = {sv_mean:.{digits_sv}f} $\\pm$ {sv_std:.{digits_sv}f}"
            f"\\end{{tabular}}"
        )

    return (
        f"\\begin{{tabular}}[c]{{@{{}}c@{{}}}}"
        f"{acc_mean:.{digits_acc}f} $\\pm$ {acc_std:.{digits_acc}f} \\\\ "
        f"{eps_mean:.{digits_eps}f} $\\pm$ {eps_std:.{digits_eps}f} \\\\ "
        f"$\\Delta\\sum \\sigma$ = {sv_mean:.{digits_sv}f} $\\pm$ {sv_std:.{digits_sv}f}"
        f"\\end{{tabular}}"
    )

def clone_model(model):
    # Deepcopy is safest because resizing mutates the module parameter structure.
    m = copy.deepcopy(model)
    m.eval()
    return m

@torch.no_grad()
def predict_logits(model, dataloader, device):
    model.eval()
    ys = []
    yhat = []
    for xb, yb in dataloader:
        xb = xb.to(device)
        out = model(xb)
        yhat.append(out.detach().cpu())
        ys.append(yb.detach().cpu())
    return torch.cat(yhat, dim=0), torch.cat(ys, dim=0)

def accuracy_from_logits(logits, labels):
    pred = logits.argmax(dim=1)
    return 100.0 * (pred == labels).float().mean().item()

def epsilon_from_logits(logits_after, logits_before):
    # epsilon = (1/n) sum_n ||y_after - y_before||_2
    diff = logits_after - logits_before
    return torch.linalg.norm(diff, dim=1).mean().item()

def singular_value_sum(model, layer):
    singular_values = model.get_singular_values(layer=layer, use_torch=True)
    return float(singular_values.detach().cpu().sum().item())

def apply_resize(model, layer, row_value):
    """
    baseline: untouched
    0: full diagonalise only
    +k: full diagonalise then k neurogenerate
    -k: full diagonalise then k neurodegenerate
    """
    resized = clone_model(model)

    if row_value == "baseline":
        return resized

    resized.full_diagonalise(layer)

    if row_value > 0:
        for _ in range(int(row_value)):
            resized.neurogenerate(layer)
    elif row_value < 0:
        for _ in range(int(abs(row_value))):
            resized.neurodegenerate(layer)

    return resized

def load_saved_model(combo, repeat_idx):
    intrinsic, correction = combo
    path = SAVE_DIR / f"repeat_{repeat_idx:02d}_{intrinsic}_{correction}_{TOTAL_EPOCHS}.pkl"

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find saved pickle:\n  {path}\n"
            f"Check SAVE_DIR, TOTAL_EPOCHS, REPEATS, and filename convention."
        )

    with open(path, "rb") as f:
        obj = pickle.load(f)

    # Expected format: {'stats': ..., 'model': ...}
    if isinstance(obj, dict) and "model" in obj:
        model = obj["model"]
        stats = obj.get("stats", None)
    else:
        model = obj
        stats = None

    return model, stats, path

def make_latex_table(df, caption=None, label=None):
    headers = list(df.columns)
    col_spec = "l" + "c" * len(headers)

    lines = []
    lines.append("\\begin{table}[ht]")
    lines.append("\\centering")
    lines.append("\\scriptsize")
    lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
    lines.append("\\hline")
    lines.append("Row & " + " & ".join(headers) + " \\\\")
    lines.append("\\hline")

    for idx, row in df.iterrows():
        cells = [str(idx)] + [row[h] for h in headers]
        lines.append(" & ".join(cells) + " \\\\")

    lines.append("\\hline")
    lines.append("\\end{tabular}")

    if caption is not None:
        lines.append(f"\\caption{{{caption}}}")
    if label is not None:
        lines.append(f"\\label{{{label}}}")

    lines.append("\\end{table}")
    return "\n".join(lines)


# -------------------------
# Main evaluation
# -------------------------
device = try_gpu()

# Expect test_loader / testing_set to already exist in the notebook.
if "testing_set" in globals():
    TEST_LOADER = testing_set
elif "test_loader" in globals():
    TEST_LOADER = test_loader
else:
    raise NameError("Could not find `testing_set` or `test_loader` in the notebook namespace.")

if not SAVE_DIR.exists():
    raise FileNotFoundError(f"SAVE_DIR does not exist: {SAVE_DIR.resolve()}")

results_nested = {}
paths_used = {}

for combo in COMBINATIONS:
    combo_key = combo_to_key(combo)
    results_nested[combo_key] = {}

    if VERBOSE:
        print(f"\n=== Combination: {combo} ===")

    # Per row, store paired per-repeat statistics
    per_row_acc_values = {row: [] for row in ROWS}
    per_row_eps_values = {row: [] for row in ROWS}
    per_row_sv_values = {row: [] for row in ROWS}

    for repeat_idx in range(REPEATS):
        base_model, base_stats, base_path = load_saved_model(combo, repeat_idx)
        paths_used[(combo_key, repeat_idx)] = str(base_path)

        base_model = clone_model(base_model).to(device)
        base_logits, base_labels = predict_logits(base_model, TEST_LOADER, device)
        base_acc = accuracy_from_logits(base_logits, base_labels)
        base_sv_sum = singular_value_sum(base_model, LAYER)

        if VERBOSE:
            print(
                f"repeat {repeat_idx:02d} | baseline acc = {base_acc:.4f}% "
                f"| baseline sum(sigma) = {base_sv_sum:.4f} | {base_path}"
            )

        for row in ROWS:
            resized_model = apply_resize(base_model, LAYER, row).to(device)
            resized_logits, resized_labels = predict_logits(resized_model, TEST_LOADER, device)
            resized_sv_sum = singular_value_sum(resized_model, LAYER)

            # Sanity: labels should match
            if not torch.equal(base_labels, resized_labels):
                raise RuntimeError("Test labels changed between baseline and resized evaluation.")

            resized_acc = accuracy_from_logits(resized_logits, resized_labels)

            if row == "baseline":
                # Baseline reports raw accuracy and original sum of singular values.
                acc_value = base_acc
                eps_value = np.nan
                sv_value = base_sv_sum
            else:
                # Paired delta within the same repeat.
                acc_value = resized_acc - base_acc
                eps_value = epsilon_from_logits(resized_logits, base_logits)
                sv_value = base_sv_sum - resized_sv_sum

            per_row_acc_values[row].append(acc_value)
            per_row_eps_values[row].append(eps_value)
            per_row_sv_values[row].append(sv_value)

            del resized_model, resized_logits, resized_labels
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        del base_model, base_logits, base_labels
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Aggregate after all repeats for this combo
    for row in ROWS:
        acc_arr = np.asarray(per_row_acc_values[row], dtype=float)
        eps_arr = np.asarray(per_row_eps_values[row], dtype=float)
        sv_arr = np.asarray(per_row_sv_values[row], dtype=float)

        results_nested[combo_key][row] = {
            "acc_mean": float(np.mean(acc_arr)),
            "acc_std": float(safe_std(acc_arr)),
            "eps_mean": float(np.nanmean(eps_arr)) if not np.all(np.isnan(eps_arr)) else np.nan,
            "eps_std": float(safe_std(eps_arr[~np.isnan(eps_arr)])) if not np.all(np.isnan(eps_arr)) else np.nan,
            "sv_mean": float(np.mean(sv_arr)),
            "sv_std": float(safe_std(sv_arr)),
            "acc_values": acc_arr,
            "eps_values": eps_arr,
            "sv_values": sv_arr,
        }


# -------------------------
# Build display DataFrame
# -------------------------
display_df = pd.DataFrame(index=[str(r) for r in ROWS])

for combo in COMBINATIONS:
    combo_key = combo_to_key(combo)
    header = combo_to_header(combo)

    cells = []
    for row in ROWS:
        stats = results_nested[combo_key][row]
        cells.append(
            format_cell(
                stats["acc_mean"], stats["acc_std"],
                stats["eps_mean"], stats["eps_std"],
                stats["sv_mean"], stats["sv_std"],
                row,
                digits_acc=3, digits_eps=4, digits_sv=4
            )
        )

    display_df[header] = cells

display(display_df)


# -------------------------
# Write LaTeX
# -------------------------
caption = (
    f"Test-set performance after resizing at layer $L={LAYER}$. "
    f"For baseline, the reported accuracy is the raw test accuracy of the original trained model, "
    f"and the second line reports the original baseline sum of singular values $\\sum \\sigma_0$. "
    f"For all other rows, the reported accuracy is the paired within-repeat change "
    f"$a_\\mathrm{{after}}-a_\\mathrm{{before}}$. "
    f"The second line in each non-baseline cell reports "
    f"$\\epsilon = \\frac{{1}}{{n}}\\sum_n \\lVert \\vec{{y}}_{{after}}-\\vec{{y}}_{{before}} \\rVert_2$ "
    f"averaged over the test set. "
    f"The third line reports the singular-value mass removed, computed per repeat as "
    f"$\\Delta\\sum \\sigma = \\sum \\sigma_0 - \\sum \\sigma_\\mathrm{{after}}$. "
    f"All uncertainties are mean $\\pm$ standard deviation over repeats."
)

latex_table = make_latex_table(
    display_df,
    caption=caption,
    label=f"tab:resize_eval_layer_{LAYER}"
)

with open(OUTPUT_TEX, "w") as f:
    f.write(latex_table)

print("\nWrote LaTeX table to:", OUTPUT_TEX)
print("\nLaTeX preview:\n")
print(latex_table)


# -------------------------
# Optional: expose raw numeric summaries too
# -------------------------
raw_rows = []
for combo in COMBINATIONS:
    combo_key = combo_to_key(combo)
    for row in ROWS:
        s = results_nested[combo_key][row]
        raw_rows.append({
            "combination": combo_to_header(combo),
            "row": row,
            "acc_mean": s["acc_mean"],
            "acc_std": s["acc_std"],
            "eps_mean": s["eps_mean"],
            "eps_std": s["eps_std"],
            "sv_removed_or_baseline_sum_mean": s["sv_mean"],
            "sv_removed_or_baseline_sum_std": s["sv_std"],
        })

raw_df = pd.DataFrame(raw_rows)
display(raw_df)

In [ ]:
OUTPUT_TEX = Path(f"./DataTables/NEW_resize_eval_layer_{LAYER}.tex")
with open(OUTPUT_TEX, "w") as f:
    f.write(latex_table)

In [ ]:
# =========================
# Plot from saved LaTeX table:
# Experiment 1: epsilon vs delta sum of singular values
# =========================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -------------------------
# Locate saved LaTeX table
# -------------------------
if "OUTPUT_TEX" in globals():
    TEX_PATH = Path(OUTPUT_TEX)
elif "LAYER" in globals():
    TEX_PATH = Path(f"resize_eval_layer_{LAYER}.tex")
else:
    TEX_PATH = Path("resize_eval_layer_1.tex")

if not TEX_PATH.exists():
    raise FileNotFoundError(
        f"Could not find LaTeX table at {TEX_PATH.resolve()}. "
        "Run the resize-evaluation table cell first."
    )

print(f"Loading LaTeX table from: {TEX_PATH.resolve()}")


# -------------------------
# Parsing helpers
# -------------------------
def parse_mean_pm_pairs(cell):
    """
    Extract all 'mean ± std' pairs from a LaTeX cell.
    Expected non-baseline Experiment 1 cell order:
        1. accuracy delta mean/std
        2. epsilon mean/std
        3. delta sum sigma mean/std
    """
    pattern = r"([+-]?\d+(?:\.\d+)?)\s*\$\\pm\$\s*([+-]?\d+(?:\.\d+)?)"
    return [(float(a), float(b)) for a, b in re.findall(pattern, cell)]


def split_latex_row(line):
    """
    Split a simple LaTeX table row on column separators.
    This works for these generated tables because nested tabular cells
    do not contain top-level '&' separators.
    """
    line = line.strip()
    if line.endswith(r"\\"):
        line = line[:-2].strip()
    return [part.strip() for part in line.split(" & ")]


# -------------------------
# Parse table
# -------------------------
tex = TEX_PATH.read_text(encoding="utf-8")

lines = []
for raw_line in tex.splitlines():
    line = raw_line.strip()
    if not line:
        continue
    if line.startswith(("\\begin", "\\end", "\\hline", "\\caption", "\\label")):
        continue
    if " & " not in line:
        continue
    lines.append(line)

header_line = next(line for line in lines if line.startswith("Row & "))
headers = split_latex_row(header_line)[1:]

records = []

for line in lines:
    if line == header_line:
        continue

    parts = split_latex_row(line)
    row_label = parts[0]

    if row_label.lower() == "baseline":
        continue

    row_value = int(row_label)

    for method, cell in zip(headers, parts[1:]):
        pairs = parse_mean_pm_pairs(cell)

        if len(pairs) < 3:
            raise ValueError(
                f"Could not parse three mean/std pairs from row={row_label}, method={method}:\n{cell}"
            )

        acc_mean, acc_std = pairs[0]
        eps_mean, eps_std = pairs[1]
        delta_sigma_mean, delta_sigma_std = pairs[2]

        records.append(
            {
                "row": row_value,
                "method": method,
                "accuracy_delta_mean": acc_mean,
                "accuracy_delta_std": acc_std,
                "epsilon_mean": eps_mean,
                "epsilon_std": eps_std,
                "delta_sum_sigma_mean": delta_sigma_mean,
                "delta_sum_sigma_std": delta_sigma_std,
            }
        )

plot_df = pd.DataFrame(records)

display(plot_df)


# -------------------------
# Plot: epsilon against delta sum sigma
# -------------------------
fig, ax = plt.subplots(figsize=(12, 8))

for method, sub in plot_df.groupby("method", sort=False):
    sub = sub.sort_values("delta_sum_sigma_mean")

    y = sub["epsilon_mean"].to_numpy()
    yerr = sub["epsilon_std"].to_numpy()

    # Log scale cannot display error bars reaching <= 0.
    yerr_lower = np.minimum(yerr, y * 0.999)
    yerr_upper = yerr

    ax.errorbar(
        sub["delta_sum_sigma_mean"],
        y,
        xerr=sub["delta_sum_sigma_std"],
        yerr=[yerr_lower, yerr_upper],
        fmt="o-",
        capsize=3,
        linewidth=1,
        markersize=4,
        label=method,
    )

ax.set_yscale("log")
ax.set_xlabel(r"$\Delta \sum \sigma$")
ax.set_ylabel(r"$\epsilon$  (log scale)")
ax.set_title(r"Experiment 1: $\epsilon$ vs. removed singular-value mass")
ax.grid(True, which="both", alpha=0.35)
ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()

out_path = TEX_PATH.with_name(f"{TEX_PATH.stem}_epsilon_vs_delta_sum_sigma_log.png")
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved plot to: {out_path.resolve()}")